# Notebook 01 – OSM-Daten Kanton Zürich laden
**Charge & Ride Züri** | ZHAW Geomarketing FS2026

Dieses Notebook lädt alle benötigten OpenStreetMap-Daten für den Kanton Zürich
und speichert sie als GeoJSON in `data/processed/`.

**Output-Files:**
- `ladestationen_zh.geojson` – Bestehende E-Bike-Ladestationen
- `gastronomie_zh.geojson` – Restaurants und Cafés
- `bahnhoefe_zh.geojson` – Bahnhöfe
- `velorouten_zh.geojson` – Offizielle Velorouten
- `kanton_zh_boundary.geojson` – Kantonsgrenze

In [ ]:
# Imports
from pathlib import Path
import time
import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt

# Projektpfade
ROOT = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# OSMnx-Konfiguration
ox.settings.log_console = True
ox.settings.use_cache = True  # Verhindert doppelte Downloads

print(f'Output-Verzeichnis: {DATA_PROCESSED}')

## 1. Kantonsgrenze laden

In [ ]:
# Kanton Zürich als Polygon laden
PLACE = 'Kanton Zürich, Switzerland'

kanton_zh = ox.geocode_to_gdf(PLACE)
kanton_zh = kanton_zh.to_crs(epsg=2056)  # In LV95 umrechnen

print(f'CRS: {kanton_zh.crs}')
print(f'Geometrie-Typ: {kanton_zh.geometry.geom_type.values[0]}')

# Als GeoJSON speichern (WGS84 für GeoJSON-Standard)
kanton_zh.to_crs(epsg=4326).to_file(DATA_PROCESSED / 'kanton_zh_boundary.geojson', driver='GeoJSON')
print('Kantonsgrenze gespeichert.')

## 2. Bestehende E-Bike-Ladestationen

In [ ]:
# OSM-Tags für E-Bike-Ladestationen
# Quelle: https://wiki.openstreetmap.org/wiki/Tag:amenity%3Dcharging_station
tags_ladestationen = {
    'amenity': 'charging_station',
    'bicycle': True  # Nur Stationen die Fahrräder/E-Bikes laden
}

# TODO: OSMnx unterstützt keine AND-Verknüpfung direkt -> Filtern nach Download
tags_ladestationen_all = {'amenity': 'charging_station'}

gdf_ladestationen = ox.features_from_place(PLACE, tags=tags_ladestationen_all)
time.sleep(1)  # Rate Limiting respektieren

# Nur Punkte behalten und nach bicycle=yes filtern (falls Attribut vorhanden)
gdf_ladestationen = gdf_ladestationen[gdf_ladestationen.geometry.geom_type == 'Point'].copy()
if 'bicycle' in gdf_ladestationen.columns:
    mask = gdf_ladestationen['bicycle'].isin(['yes', 'designated', 'True', True])
    gdf_ladestationen_ebike = gdf_ladestationen[mask].copy()
    print(f'E-Bike-Ladestationen (gefiltert): {len(gdf_ladestationen_ebike)}')
else:
    gdf_ladestationen_ebike = gdf_ladestationen.copy()
    print(f'Alle Ladestationen (kein bicycle-Attribut): {len(gdf_ladestationen_ebike)}')

# In LV95 umrechnen und speichern
gdf_ladestationen_ebike = gdf_ladestationen_ebike.to_crs(epsg=2056)
gdf_ladestationen_ebike[['geometry', 'name', 'amenity']].to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'ladestationen_zh.geojson', driver='GeoJSON'
)
print('Ladestationen gespeichert.')

## 3. Gastronomie (Restaurants und Cafés)

In [ ]:
# Gastronomie-POIs laden
tags_gastro = {'amenity': ['restaurant', 'cafe']}

gdf_gastro = ox.features_from_place(PLACE, tags=tags_gastro)
time.sleep(1)

# Nur Punkte (Polygone zu Zentroiden umrechnen)
gdf_gastro_punkte = gdf_gastro.copy()
mask_poly = gdf_gastro_punkte.geometry.geom_type != 'Point'
gdf_gastro_punkte.loc[mask_poly, 'geometry'] = gdf_gastro_punkte[mask_poly].geometry.centroid

gdf_gastro_punkte = gdf_gastro_punkte.to_crs(epsg=2056)
print(f'Gastronomie-POIs: {len(gdf_gastro_punkte)}')

# Relevante Spalten behalten
cols = ['geometry', 'name', 'amenity']
cols_existing = [c for c in cols if c in gdf_gastro_punkte.columns]
gdf_gastro_punkte[cols_existing].to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'gastronomie_zh.geojson', driver='GeoJSON'
)
print('Gastronomie gespeichert.')

## 4. Bahnhöfe

In [ ]:
# Bahnhöfe laden
tags_bahnhoefe = {'railway': 'station'}

gdf_bahnhoefe = ox.features_from_place(PLACE, tags=tags_bahnhoefe)
time.sleep(1)

# Nur Punkte behalten
gdf_bahnhoefe = gdf_bahnhoefe[gdf_bahnhoefe.geometry.geom_type == 'Point'].copy()
gdf_bahnhoefe = gdf_bahnhoefe.to_crs(epsg=2056)
print(f'Bahnhöfe: {len(gdf_bahnhoefe)}')

cols = ['geometry', 'name', 'railway']
cols_existing = [c for c in cols if c in gdf_bahnhoefe.columns]
gdf_bahnhoefe[cols_existing].to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'bahnhoefe_zh.geojson', driver='GeoJSON'
)
print('Bahnhöfe gespeichert.')

## 5. Übersichtsplot

In [ ]:
import contextily as cx

fig, ax = plt.subplots(figsize=(12, 10))

kanton_zh.boundary.plot(ax=ax, color='black', linewidth=1.5, zorder=3)
gdf_gastro_punkte.plot(ax=ax, color='orange', markersize=2, alpha=0.5, label='Gastronomie', zorder=4)
gdf_bahnhoefe.plot(ax=ax, color='blue', markersize=8, marker='s', label='Bahnhöfe', zorder=5)
gdf_ladestationen_ebike.to_crs(epsg=2056).plot(ax=ax, color='green', markersize=10, marker='*', label='E-Bike-Ladestationen', zorder=6)

cx.add_basemap(ax, crs=kanton_zh.crs, source=cx.providers.OpenStreetMap.Mapnik, alpha=0.5)

ax.legend(fontsize=10)
ax.set_title('Kanton Zürich – OSM-Datenübersicht', fontsize=14)
ax.set_axis_off()

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'uebersicht_osm_daten.png', dpi=150, bbox_inches='tight')
plt.show()
print('Übersichtsplot gespeichert.')

## Zusammenfassung

| Datensatz | Anzahl POIs | File |
|---|---|---|
| E-Bike-Ladestationen | - | ladestationen_zh.geojson |
| Gastronomie | - | gastronomie_zh.geojson |
| Bahnhöfe | - | bahnhoefe_zh.geojson |

→ Weiter mit Notebook 02